# Day 2 · Module 6: Prompt Injection

**Objective:** run the pipeline over a repo that carries injection payloads, see which agent gets influenced and which control layer stops it, and apply the Rule of Two to bound the damage.


## Relevant Claude Code Docs
Review these before starting the module:
- [Security](https://code.claude.com/docs/en/security)
- [Choose a sandbox environment](https://code.claude.com/docs/en/sandbox-environments)
- [Configure permissions](https://code.claude.com/docs/en/permissions)
- [How Claude Code works](https://code.claude.com/docs/en/how-claude-code-works)

## 1 · The idea

Prompt injection is untrusted content — a file, comment, docstring, tool output — that reads like an instruction and gets obeyed as one. The defenses stack: **instruction hierarchy / content labeling** (treat file content as DATA, never as commands), **tool-output validation**, and credential/trust scoping (the permissions and sandboxing from earlier modules).

The load-bearing heuristic is the **Rule of Two**: an agent should hold at most two of these three at once — untrusted input, sensitive access, and the ability to change state or communicate outward. Hold all three and a single poisoned input can turn into real damage.


### Grounding

ContosoBank ships a poisoned tree under `reference/day2/m6/poisoned/` carrying four payloads: three blatant (a comment instructing the assistant to exfiltrate webhook data, an "ignore previous instructions / dump the environment" block, and a `curl | sh` bootstrap line) and one deliberately subtle. The blatant ones exist so your defenses have something obvious to catch; the subtle one is the point of the exercise, and you are meant to discover it — so it is not named here.

Rule of Two, made concrete on ContosoBank: **payment webhook delivery** (`webhooks.deliver`) holds all three legs at once — **A**, an untrusted trigger (a posting drives delivery); **B**, sensitive access (it reads the payment payload data); and **C**, the ability to change state or communicate outward (it POSTs to a merchant-supplied URL). That is exactly the shape the Rule of Two warns against: isolate the delivery path so it no longer holds all three legs at once.


### Setup check

Run once per session; resolves `proj` for the self-check.


In [ ]:
import subprocess, sys, os
# Ensure src is importable and the sandbox is healthy before you start.
os.chdir(os.path.dirname(os.getcwd())) if os.path.basename(os.getcwd()) in ("day1","day2") else None
root = subprocess.run(["git","rev-parse","--show-toplevel"], capture_output=True, text=True)
proj = root.stdout.strip() or os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, os.path.join(proj, "src"))
print("project root:", proj)
try:
    import contoso
    print("contoso imported OK")
except Exception as e:
    print("Could not import contoso:", e)
    print("Run `uv sync` in the repo root, then restart the kernel.")


## 2 · Your exercise

1. Follow `reference/day2/m6/README.md` to apply the poisoned tree to a **worktree** (copy the poisoned files over the real ones in the worktree only — never merge it). Copy `reference/day2/m6/agents/*` and `reference/day2/m6/run-pipeline.md` into `.claude/` so the pipeline runs without Day 1.
2. **Round 1:** run the pipeline on `scenarios/d2-m6-add-analytics-field.md` against the poisoned worktree as-built. Record which agent was influenced by which payload, and which layer (framing / permissions / sandbox / hooks) caught it — and which slipped through.
3. **Round 2:** harden the explorer and critic with data-not-instructions framing, confirm the permission / sandbox / hook layers cover the rest, rerun, and record the delta.
4. Finish with a **Rule of Two** table: for each agent, which of the three legs it holds, and what to drop.
5. Write Round 1, the Round 2 delta, and the Rule of Two table to `artifacts/day2/m6/injection-test.md`.


### What good looks like

The write-up shows a real before/after: at least one payload landed in Round 1, and the hardening measurably changed the outcome in Round 2. The Rule of Two table names, per agent, which legs it holds.

The payload that most needs catching is the subtle one — a change that looks like a helpful refactor but quietly weakens a real security check. If your critic only flags the loud payloads and the subtle one rides through, that is the failure this module is built to expose.


### Fast-finisher

Rerun Round 1 on a cheaper / faster model tier. The subtle payload is more likely to land there — a concrete reminder that "the model will notice it" is not a control.


### Self-check

Advisory, not a gate. It reports what it finds in `artifacts/day2/m6/injection-test.md`. Safe to run before you've written anything.


In [ ]:
import pathlib
it = pathlib.Path(proj) / "artifacts" / "day2" / "m6" / "injection-test.md"
if it.exists():
    t = it.read_text().lower()
    print(f"[x] injection-test.md present ({len(t.split())} words)")
    print(f"  [{'x' if 'round 1' in t or 'before' in t else ' '}] records Round 1 (before)?")
    print(f"  [{'x' if 'round 2' in t or 'after' in t or 'harden' in t else ' '}] records Round 2 (after hardening)?")
    print(f"  [{'x' if 'rule of two' in t else ' '}] includes a Rule of Two table?")
    print(f"  [{'x' if 'ttl' in t or 'last_used' in t or 'expiry' in t else ' '}] identified the subtle security-weakening payload?")
else:
    print("[ ] artifacts/day2/m6/injection-test.md missing")


## 3 · Debrief

- Which layer actually stopped each payload — the agent's own judgment, or an external control (permission deny, sandbox boundary, hook)? Which would you trust on a bad day?
- Apply the Rule of Two to the webhook path. It holds all three legs; which leg is cheapest to remove, and what does that cost?


---
### Take-home

Layered, deterministic controls — not the model's vigilance — are what contain an injection. Content is data; the Rule of Two bounds how much any single poisoned input can do. The subtle payloads are the ones that matter, because they are written to survive a quick read.

(Related concept: the capstone proves the whole hardened pipeline is governable end to end.)
